### Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import inspect
from scipy.special import lambertw

### Формулы самих методик для подбора коэффициентов

In [ ]:
def nazarov_sipachev(X, a, b, lamda):
    V_l, V_o = X # распаковка массива фактических данныъ в добычу жидкости и нефти
    V_w = V_l - V_o # выичсление добычи воды
    return V_l / V_o - a - b * V_w # перенос в левую часть

def kambarov(X, a, b, lamda):
    V_l, V_o = X # распаковка массива фактических данныъ в добычу жидкости и нефти
    return V_o - a + b / V_l

def ln_kambarov(X, a, b, lamda):
    V_l, V_o = X # распаковка массива фактических данныъ в добычу жидкости и нефти
    return np.log(V_o) - a + b / np.log(V_l)

def pirverdyan(X, a, b, lamda):
    V_l, V_o = X
    return V_o - a + b / np.sqrt(V_l)

def gaysin(X, a, b, lamda):
    V_l, V_o = X
    return V_o - a - b * V_o / V_l

def Stasenkov_Rakhimkulov_Rudchuk():
    return None

def kazakov(X, a, b, lamda):
    V_l, V_o = X
    return V_o - a - b * np.power(V_l, - lamda)

def mod_Nazarov_Sipachev(X, a, b, lamda):
    V_l, V_o = X
    V_w = V_l - V_o
    return np.log(V_w / V_o) - a - b * V_o

def mod_Sipachev_Posevich(X, a, b, lamda):
    V_l, V_o = X
    return np.log(V_o / V_l) - a + b * V_o

def Abizbaev(X, a, b, lamda):
    V_l, V_o = X
    return np.log(V_o) - a - b * np.log(V_l)

def foil_const(X, a, b, lamda):
    V_l, V_o = X
    return V_o - a - b * V_l


XB_funcs_dict = {nazarov_sipachev.__name__: nazarov_sipachev,
                 kambarov.__name__: kambarov, 
                 ln_kambarov.__name__: ln_kambarov,
                 pirverdyan.__name__: pirverdyan, 
                 gaysin.__name__: gaysin,
                 kazakov.__name__: kazakov,
                 mod_Nazarov_Sipachev.__name__: mod_Nazarov_Sipachev,
                 mod_Sipachev_Posevich.__name__: mod_Sipachev_Posevich,
                 Abizbaev.__name__: Abizbaev,
                 foil_const.__name__: foil_const}

### Формула оизов по методикам

In [ ]:

def OIZ_nazarov_sipachev(Q_liq, a, b, lamda, q_o_pr=0.5):
    lamda = lamda
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициенты методики (подбираются аппроксимацией фактических данных)
    """
    D = (a + b * (Q_liq )) ** 2 - 4 * b * (Q_liq)
    if D > 0:
        return min((a + b * (Q_liq) + np.sqrt(D)) / (2 * b), (a + b * (Q_liq) - np.sqrt(D)) / (2 * b))
    return 0


def OIZ_kambarov(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/сут
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a - b / (Q_liq)


def OIZ_ln_kambarov(Q_liq, a=0, b=0, lamda=0, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return np.exp(a - b / (np.log(Q_liq)))


def OIZ_Pirverdyan(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a - b / np.sqrt(Q_liq)



def OIZ_gaysin(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a / (1 - b / (Q_liq))


def OIZ_Stasenkov_Rakhimkulov_Rudchuk():
    return None

def OIZ_kazakov(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b, lamda - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a + b * (Q_liq) ** (- lamda)


def OIZ_Sazonov(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a - b * np.log(Q_liq)


def OIZ_Sipachev_Posevich(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return (Q_liq) / (a + b * (Q_liq))


def OIZ_mod_Nazarov_Sipachev(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return (Q_liq) - 1 / b * lambertw( - b / (Q_liq) * np.exp(a + b * (Q_liq))).real
    
    
def OIZ_mod_Sipachev_Posevich(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return 1 / b * lambertw(b * (Q_liq) * np.exp(a)).real
    # return q_o_pr * (Q_liq + q_liq * t_pr) / (q_liq - b * q_o_pr * (Q_liq + q_liq * t_pr))

def OIZ_Abizbaev(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return np.exp(a + b * np.log((Q_liq)))
    # return q_o_pr * (Q_liq + q_liq * t_pr) / (b * q_liq)

def OIZ_foil_const(Q_liq, a, b, lamda, q_o_pr=0.5):
    """
    q_liq - дебит жидкости на последнюю дату, м3/мес
    Q_liq - накопленная добыча жидкости на последнюю дату, м3
    t_pr - предельно достижимое время, мес. (зависит от того в каких координатах аппроксимировал темпы падения. Скорее всего это буду месяцы)
    q_o_pr - предельно рентабельный дебит нефти, т/мес (по умолчанию 0.5)
    a, b - коэффициент методики (подбираются аппроксимацией фактических данных)
    """
    return a + b * (Q_liq)

XB_OIZ_funcs_dict = {nazarov_sipachev.__name__: OIZ_nazarov_sipachev,
                 kambarov.__name__: OIZ_kambarov, 
                 ln_kambarov.__name__: OIZ_ln_kambarov,
                 pirverdyan.__name__: OIZ_Pirverdyan, 
                 gaysin.__name__: OIZ_gaysin,
                 kazakov.__name__: OIZ_kazakov,
                 mod_Nazarov_Sipachev.__name__: OIZ_mod_Nazarov_Sipachev,
                 mod_Sipachev_Posevich.__name__: OIZ_mod_Sipachev_Posevich,
                 Abizbaev.__name__: OIZ_Abizbaev,
                 foil_const.__name__: OIZ_foil_const}

### Аппроксимация темпов падения по Arps

In [ ]:
def arps_func(t, d, b):
    return np.exp(- 1 / b * np.log(1 + b * d * t))

### Загрузка эксель файла

In [ ]:
wells = pd.read_excel('Скважины для модели.xlsx')
name = 'Первый расчёт'
wells_list_path = r'C:\Users\pogre\Desktop\Проект для НТК\Список скважин.txt'
grp_path = r'C:\Users\pogre\Desktop\Проект для НТК\Даты повторного ГРП.xlsx'

### Подбор коэффициентов

In [ ]:
from scipy.optimize import least_squares

def fit_coefs(wells, name, wells_list_path = None):
    global well_names
    if wells_list_path:
        with open(wells_list_path, encoding='utf-8') as f:
            valid_wells = set(line.strip() for line in f if line.strip())
        wells = wells[wells['№ скважины'].astype(str).isin(valid_wells)]

    wells = wells[
        (wells['Накопленный отбор жидкости, т'] != 0)]
    well_names = wells['№ скважины'].unique()

    # Функция для подбора коэффициентов
    def fit_coefficients(func, V_l, V_o):
        """Подбирает a, b, lamda для заданной функции."""
        X = (V_l[-12:], V_o[-12:])
        
        # Определяем, какие параметры нужны (есть ли lamda?)
        has_lamda = "lamda" in func.__code__.co_varnames
        
        # Начальные guess-значения
        initial_guess = [1.0, 1.0]
        if has_lamda:
            initial_guess.append(1.0)
        
        # Целевая функция для оптимизации (минимизация f(X, a, b, lamda) → 0)
        def target(params):
            a, b = params[:2]
            lamda = params[2] if has_lamda else 0.0
            return func(X, a, b, lamda)
        
        # Оптимизация
        result = least_squares(target, initial_guess)
        
        # Возвращаем коэффициенты
        a, b = result.x[:2]
        lamda = result.x[2] if has_lamda else np.nan
        # error = np.mean(np.abs(target(result.x)))  # Средняя ошибка
        error = np.sqrt(np.mean(target(result.x) ** 2))  # СКРО (RMSE)
        return a, b, lamda, error

    # Создаём DataFrame для результатов
    results = []

    # Проходим по всем скважинам и моделям
    for well in well_names:
        well_data = wells[wells['№ скважины'] == well]
        V_l = well_data['Накопленный отбор жидкости, т'].values
        V_o = well_data['Накопленный отбор нефти, т'].values
        
        for model_name, model_func in XB_funcs_dict.items():
            try:
                a, b, lamda, error = fit_coefficients(model_func, V_l, V_o)
                results.append({
                    'Скважина': well,
                    'Модель': model_name,
                    'a': a,
                    'b': b,
                    'lamda': lamda,
                    'Ошибка': error
                })
            except Exception as e:
                print(f"Ошибка для скважины {well}, модели {model_name}: {e}")

    # Сохраняем в DataFrame
    results_df = pd.DataFrame(results)
    df_grouped = results_df.set_index(["Скважина", "Модель"])
    results_df.to_excel(f"Результаты подбора коэффициентов для {name}.xlsx", index=False)
    return results_df
results_df = fit_coefs(wells=wells, name = name, wells_list_path=wells_list_path)

### Снятие темпов по жидкости и расчет базы

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def extract_valid_series(series, window=12):
    series = series[-12:].dropna().values
    valid_q = []
    for i in range(1, len(series)):
        if series[i - 1] > 0 and 1 / 2 <= series[i] / series[i - 1] <= 2 and series[i - 1] is not np.nan:
            valid_q.append(series[i - 1])
        else:
            valid_q = []
    if valid_q:
        return valid_q
    return []

def calculate_q_liq_forecast_arps_from_file(wells, wells_list_path, months=312):
    wells['Дата'] = pd.to_datetime(wells['Дата'], dayfirst=True)

    with open(wells_list_path, 'r', encoding='utf-8') as f:
        well_ids = [line.strip() for line in f if line.strip()]

    result = []

    for well_id in well_ids:
        well_data = wells[wells['№ скважины'].astype(str) == well_id].sort_values('Дата')
        q_series = well_data['Дебит жидкости за последний месяц, т/сут']

        valid_series = extract_valid_series(q_series, window=12)

        if len(valid_series) < 2:
            print(f"Недостаточно валидных точек для скважины {well_id}. Пропускаем.")
            continue

        q0 = valid_series[0]
        decline_factors = np.array([q / q0 for q in valid_series])

        try:
            t_data = np.arange(len(decline_factors))
            popt, _ = curve_fit(arps_func, t_data, decline_factors, bounds=([1e-6, 1e-6], [30, 30]))
            d_opt, b_opt = popt
        except RuntimeError:
            print(f"Не удалось аппроксимировать темпы для скважины {well_id}. Пропускаем.")
            continue

        # Полный прогноз темпов
        forecast_t = np.arange(0, months + len(decline_factors))
        forecast_temp_full = arps_func(forecast_t, d_opt, b_opt)

        # Пересчёт "мнимого нулевого" дебита
        last_q_liq = valid_series[-1]
        temp_at_last = forecast_temp_full[len(decline_factors) - 1]
        q0_fake = last_q_liq / temp_at_last

        # Прогноз с t начиная с len(decline_factors)
        forecast_q_liq = q0_fake * forecast_temp_full[len(decline_factors):len(decline_factors) + months]

        for month, q in enumerate(forecast_q_liq, 1):
            result.append({
                'Скважина': well_id,
                'Месяц': month,
                'Прогноз дебита жидкости, т/сут': q
            })

    df_result = pd.DataFrame(result)
    df_result.to_excel("Прогноз_дебита_жидкости_ARPS_исправленный.xlsx", index=False)
    return df_result

df_liquid_prognoz_baza = calculate_q_liq_forecast_arps_from_file(
    wells=wells,
    wells_list_path=wells_list_path,
    months=312
)


### Расчет дебитов по месяцам

In [ ]:
import pandas as pd
import numpy as np

def calculate_weighted_debit_dynamics(wells, results_df, XB_OIZ_funcs_dict, df_forecast, months=12, wells_list_path=None):
    wells['Дата'] = pd.to_datetime(wells['Дата'], dayfirst=True)
    wells['№ скважины'] = wells['№ скважины'].astype(str)
    results_df['Скважина'] = results_df['Скважина'].astype(str)
    df_forecast['Скважина'] = df_forecast['Скважина'].astype(str)

    if wells_list_path is not None:
        with open(wells_list_path, 'r', encoding='utf-8') as f:
            wells_filter = [line.strip() for line in f if line.strip()]
        wells = wells[wells['№ скважины'].astype(str).isin(wells_filter)]
        results_df = results_df[results_df['Скважина'].astype(str).isin(wells_filter)]

    last_rows = wells.sort_values('Дата').groupby('№ скважины').tail(1)

    all_results = []

    for skv, group in results_df.groupby('Скважина'):
        well_data = last_rows[last_rows['№ скважины'] == skv]
        if well_data.empty:
            print(f"Скважина {skv} не найдена!")
            continue

        try:
            # q_liq = float(well_data['Дебит жидкости за посл.месяц, м3/сут'].iloc[0])
            Q_liq_0 = float(well_data['Накопленный отбор жидкости, т'].iloc[0])
            fact_q_oil = float(well_data['Дебит нефти за последний месяц, т/сут'].iloc[0])
        except Exception as e:
            print(f"Ошибка чтения параметров для скважины {skv}: {e}")
            continue
        
        forecast_q_liq = df_forecast[df_forecast['Скважина'] == skv].sort_values('Месяц')
        q_liq_list = forecast_q_liq['Прогноз дебита жидкости, т/сут'].tolist()

        # Если данных по Арпсу не хватает, то дополняем последним значением
        if len(q_liq_list) < months and len(q_liq_list) > 0:
            q_liq_list += [q_liq_list[-1]] * (months - len(q_liq_list))  # дополняем последним значением
        if len(q_liq_list) == 0:
            continue

        
        # Словарь дебитов по моделям
        monthly_debits_by_model = {}

        for _, row in group.iterrows():

            model_name = row['Модель']
            model_func = XB_OIZ_funcs_dict.get(model_name)
            if model_func is None:
                print(f"Модель {model_name} не найдена в словаре XB_OIZ_funcs_dict!")
                continue

            a = float(row['a'])
            b = float(row['b'])
            lamda = float(row['lamda'])

            # Перерасчёт НИЗ(0)
            niz_0 = model_func(
                Q_liq=Q_liq_0,
                a=a,
                b=b,
                lamda=lamda,
                q_o_pr=0.5
            )
            niz_values = [niz_0]

            # Расчёт НИЗ(n) и дебитов
            for m in range(1, months + 1):
                t_days = m * 30
                q_liq_m = q_liq_list[m - 1]
                Q_liq_m = Q_liq_0 + q_liq_m * t_days
                niz_curr = model_func(
                    Q_liq=Q_liq_m,
                    a=a,
                    b=b,
                    lamda=lamda,
                    q_o_pr=0.5
                )
                niz_values.append(niz_curr)

            # Дебиты: (НИЗ(n) - НИЗ(n-1)) / 30
            debits = [(niz_values[m] - niz_values[m - 1]) / 30 for m in range(1, months + 1)]
            monthly_debits_by_model[model_name] = debits



        if not monthly_debits_by_model:
            continue

        # Средневзвешенные дебиты по месяцам
        weighted_debits = []
        for m in range(months):
            month_debits = np.array([debits[m] for debits in monthly_debits_by_model.values()])
            median_debit = np.median(month_debits)
            # Оценка качества модели: отклонение от медианы
            weights = []
            for model_name, debits in monthly_debits_by_model.items():
                dev = abs(debits[m] - median_debit)
                model_row = group[group['Модель'] == model_name]

                if model_row.empty:
                    weight = 1e-3  # Маленький вес, если модель не найдена
                else:
                    appr_err = float(model_row['Ошибка'].values[0])
                    weight = 1 / (1 + dev + appr_err)

                weights.append(weight)

            weights = np.array(weights)
            weights /= weights.sum()
            weighted_debit = np.sum([
                debits[m] * w for debits, w in zip(monthly_debits_by_model.values(), weights)
            ])
            weighted_debits.append(weighted_debit)

        # Этап 2: темпы падения temp(n) = q(0) / q(n)
        q0 = weighted_debits[0]
        decline_factors = [q0 / q if q != 0 else 1 for q in weighted_debits]

        # Итоговая динамика = fact_q_oil * (1 / temp(n)) = fact_q_oil * (q(n)/q(0))
        final_dynamics = [fact_q_oil if fact_q_oil > 0.5 else 0 for fact_q_oil in [fact_q_oil * (1 / temp) for temp in decline_factors] ]

        for m, debit in enumerate(final_dynamics, 0):
            all_results.append({
                'Скважина': skv,
                'Месяц': m,
                'Прогноз дебита нефти, т/сут': debit
            })

    df_results = pd.DataFrame(all_results)
    df_results.to_excel("Прогноз_дебита_нефти_гладкий.xlsx", index=False)
    return df_results

df_oil_prognoz_baza = calculate_weighted_debit_dynamics(
    wells=wells,
    results_df=results_df,
    XB_OIZ_funcs_dict=XB_OIZ_funcs_dict,
    df_forecast=df_liquid_prognoz_baza,
    months= 25 * 12, 
    wells_list_path=wells_list_path
)


### Расчёт ОИЗ и НИЗ

In [ ]:
import pandas as pd
import numpy as np

final_results = []

wells['Дата'] = pd.to_datetime(wells['Дата'], dayfirst=True)
last_rows = wells.sort_values('Дата').groupby('№ скважины').tail(1)

for well, group in df_oil_prognoz_baza.groupby('Скважина'):
    valid_debits = group['Прогноз дебита нефти, т/сут']


    # Суммарная добыча за 25 лет по активным месяцам
    future_oil = np.sum(valid_debits * 30)  # 30 дней в месяце

    # Получаем фактическую накопленную добычу нефти на последнюю дату
    well_row = last_rows[last_rows['№ скважины'] == well]
    if not well_row.empty:
        try:
            Q_oil = float(well_row['Накопленный отбор нефти, т'].values[0])
        except Exception as e:
            print(f"Ошибка чтения Q_oil для скв. {well}: {e}")
            Q_oil = None
    else:
        Q_oil = None

    if Q_oil is not None:
        OIZ = future_oil
        NIZ = Q_oil + OIZ
    else:
        OIZ = None
        NIZ = None

    final_results.append({
        'Скважина': well,
        'Средневзвешенный НИЗ': NIZ,
        'Q_oil': Q_oil,
        'ОИЗ': OIZ
    })

final_df = pd.DataFrame(final_results)
final_df.to_excel("ОИЗ_и_НИЗ_по_прогнозу.xlsx", index=False)


### Расчёт темпов нефти и жидкости окружающих рефраков

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from scipy.optimize import curve_fit

def analyze_grp_effect_individual(wells_df, wells_list_path, grp_path, search_radius=2000, output_dir="Результаты_по_скважинам", mode="oil"):
    assert mode in ["oil", "liq"], "mode должен быть 'oil' или 'liq'"

    debit_col = 'Дебит нефти за последний месяц, т/сут' if mode == "oil" else 'Дебит жидкости за последний месяц, т/сут'
    output_dir = Path(output_dir) 
    output_dir.mkdir(parents=True, exist_ok=True)

    wells_df = wells_df.copy()
    wells_df['№ скважины'] = wells_df['№ скважины'].astype(str).str.strip().str.upper()
    wells_df['Дата'] = pd.to_datetime(wells_df['Дата'], dayfirst=True, errors='coerce')

    with open(wells_list_path, encoding='utf-8') as f:
        target_wells = [line.strip().upper() for line in f if line.strip()]

    grp_df = pd.read_excel(grp_path)
    grp_df['Скважина'] = grp_df['Скважина'].astype(str).str.strip().str.upper()
    grp_df['Дата повторного ГРП'] = pd.to_datetime(grp_df['Дата повторного ГРП'], dayfirst=True, errors='coerce')

    arps_params_rows = []

    for well_num in target_wells:
        current_well = wells_df[wells_df['№ скважины'] == well_num]
        if current_well.empty:
            print(f"Скважина {well_num} не найдена в wells_df")
            continue

        x0 = current_well['Координата X'].iloc[0]
        y0 = current_well['Координата Y'].iloc[0]

        local_radius = search_radius
        nearby_wells = []

        # Повторный поиск с увеличением радиуса
        wells_unique = wells_df.dropna(subset=['Координата X', 'Координата Y']) \
                               .groupby('№ скважины')[['Координата X', 'Координата Y']].first().reset_index()

        while True:
            wells_unique['distance'] = np.sqrt((wells_unique['Координата X'] - x0) ** 2 + (wells_unique['Координата Y'] - y0) ** 2)
            nearby_wells = wells_unique[ 
                (wells_unique['distance'] <= local_radius) & 
                (wells_unique['№ скважины'] != well_num)
            ]['№ скважины'].values

            nearby_wells_with_grp = [w for w in nearby_wells if w in grp_df['Скважина'].values]
            
            valid_neighbors = []
            for neighbor in nearby_wells_with_grp:
                grp_date = grp_df.loc[grp_df['Скважина'] == neighbor, 'Дата повторного ГРП'].iloc[0]
                if pd.isna(grp_date):
                    continue

                # Найдём первый день месяца, в котором был ГРП (напр., 08.03.2019 → 01.03.2019)
                grp_month_start = grp_date.replace(day=1)

                neighbor_data = wells_df[wells_df['№ скважины'] == neighbor].copy().sort_values('Дата')

                # Берём все данные, начиная с месяца ГРП (включительно)
                post_grp_data = neighbor_data[neighbor_data['Дата'] >= grp_month_start]
                if post_grp_data.empty:
                    continue

                date_limit = grp_date + pd.DateOffset(months=4)
                max_debit_period = post_grp_data[post_grp_data['Дата'] <= date_limit]
                if max_debit_period.empty:
                    continue

                start_idx = max_debit_period[debit_col].idxmax()
                start_value = max_debit_period.loc[start_idx, debit_col]
                start_date = max_debit_period.loc[start_idx, 'Дата']

                debits_series = [start_value]
                previous_value = start_value

                for _, row in post_grp_data[post_grp_data['Дата'] > start_date].iterrows():
                    debit = row[debit_col]
                    if pd.isna(debit) or debit == 0 or debit > start_value:
                        break
                    if not (0.1 * previous_value <= debit <= 3 * previous_value):
                        break
                    debits_series.append(debit)
                    previous_value = debit

                if len(debits_series) > 1:
                    valid_neighbors.append((neighbor, debits_series))

            # Если для соседа валидных точек меньше 2, увеличиваем радиус
            valid_neighbors = [(neighbor, debits_series) for neighbor, debits_series in valid_neighbors if len(debits_series) > 1]


            if len(valid_neighbors) < 2:
                local_radius += 2000
                print(f"Увеличен радиус до {local_radius} для скважины {well_num}")
            else:
                break  # Достаточно валидных соседей

        all_temps = []
        debits_rows = []

        # Перебираем валидных соседей
        for neighbor, _ in valid_neighbors:
            grp_date = grp_df.loc[grp_df['Скважина'] == neighbor, 'Дата повторного ГРП'].iloc[0]
            if pd.isna(grp_date):
                continue

            # Найдём первый день месяца, в котором был ГРП (напр., 08.03.2019 → 01.03.2019)
            grp_month_start = grp_date.replace(day=1)

            neighbor_data = wells_df[wells_df['№ скважины'] == neighbor].copy().sort_values('Дата')

            # Берём все данные, начиная с месяца ГРП (включительно)
            post_grp_data = neighbor_data[neighbor_data['Дата'] >= grp_month_start]
            if post_grp_data.empty:
                continue

            date_limit = grp_date + pd.DateOffset(months=4)
            max_debit_period = post_grp_data[post_grp_data['Дата'] <= date_limit]
            if max_debit_period.empty:
                continue

            start_idx = max_debit_period[debit_col].idxmax()
            start_value = max_debit_period.loc[start_idx, debit_col]
            start_date = max_debit_period.loc[start_idx, 'Дата']

            debits_series = [start_value]
            previous_value = start_value

            for _, row in post_grp_data[post_grp_data['Дата'] > start_date].iterrows():
                debit = row[debit_col]
                if pd.isna(debit) or debit == 0 or debit > start_value:
                    break
                if not (0.1 * previous_value <= debit <= 3 * previous_value):
                    break
                debits_series.append(debit)
                previous_value = debit

            if len(debits_series) > 1:
                temp_series = [q / debits_series[0] for q in debits_series]
                all_temps.append(temp_series)

                row = {'Скважина': neighbor}
                for i, val in enumerate(debits_series):
                    row[f'Debit_{i}'] = val
                debits_rows.append(row)

        if all_temps:
            max_len = max(len(t) for t in all_temps)
            temps_array = np.full((len(all_temps), max_len), np.nan)
            for i, t in enumerate(all_temps):
                temps_array[i, :len(t)] = t

            avg_temps = np.nanmean(temps_array, axis=0)
            t_vals = np.arange(len(avg_temps))

            try:
                popt, _ = curve_fit(arps_func, t_vals, avg_temps, bounds=([1e-6, 1e-6], [30, 30]))
                d_fit, b_fit = popt
            except RuntimeError:
                d_fit, b_fit = np.nan, np.nan

            df_temps = pd.DataFrame({'Месяц': t_vals, 'Темп падения (ср. арифм.)': avg_temps})
            df_debits = pd.DataFrame(debits_rows)

            file_path = output_dir / f"{well_num}.xlsx"
            with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
                df_debits.to_excel(writer, sheet_name='Дебиты_окружения', index=False)
                df_temps.to_excel(writer, sheet_name='Темпы_падения', index=False)

            arps_params_rows.append({'Скважина': well_num, 'd': d_fit, 'b': b_fit})
            print(f"Сохранено: {file_path}")
        else:
            print(f"Для скважины {well_num} не выполнен расчёт.")

    if arps_params_rows:
        df_params = pd.DataFrame(arps_params_rows)
        df_params.to_excel(output_dir / "Arps_параметры.xlsx", index=False)
        print(f"Сохранены параметры Arps: {output_dir / 'Arps_параметры.xlsx'}")

df_liquid_temps = analyze_grp_effect_individual(
    wells_df=wells,
    wells_list_path=wells_list_path,
    grp_path=grp_path,
    search_radius=2000,
    output_dir="Темпы окружения по жидкости",
    mode="liq"  # или "oil"
)



df_oil_temps = analyze_grp_effect_individual(
    wells_df=wells,
    wells_list_path=wells_list_path,
    grp_path=grp_path,
    search_radius=2000,
    output_dir="Темпы окружения по нефти",
    mode="oil"  # или "oil"
)


## ИИ Для нахождения запускного дебита

In [ ]:
import pandas as pd
import numpy as np

import pandas as pd

wells_df = pd.read_excel("Скважины для модели.xlsx")
grp_df = pd.read_excel("Даты повторного ГРП.xlsx")
params_df = pd.read_excel("Параметры ГРП.xlsx")


def get_actual_q0_after_grp(wells_df, grp_df, params_df):
    wells_df = wells_df.copy()
    wells_df['Дата'] = pd.to_datetime(wells_df['Дата'], dayfirst=True)
    wells_df['Скважина'] = wells_df['№ скважины'].astype(str).str.strip().str.upper()
    grp_df['Скважина'] = grp_df['Скважина'].astype(str).str.strip().str.upper()
    grp_df['Дата повторного ГРП'] = pd.to_datetime(grp_df['Дата повторного ГРП'], dayfirst=True)
    params_df['Скважина'] = params_df['Скважина'].astype(str).str.strip().str.upper()

    results = []

    for _, row in grp_df.iterrows():
        skv = row['Скважина']
        grp_date = row['Дата повторного ГРП']

        skv_data = wells_df[wells_df['Скважина'] == skv].sort_values('Дата')
        if skv_data.empty:
            continue

        # Последний ненулевой дебит до ГРП
        before_grp = skv_data[skv_data['Дата'] < grp_date]
        before_grp = before_grp[before_grp['Дебит нефти за последний месяц, т/сут'] > 0]
        if before_grp.empty:
            continue
        q_before = before_grp['Дебит нефти за последний месяц, т/сут'].iloc[-1]

        # Максимум после ГРП (в пределах 4 мес)
        after_grp = skv_data[skv_data['Дата'] >= grp_date]
        after_grp = after_grp[after_grp['Дата'] <= grp_date + pd.DateOffset(months=4)]
        if after_grp.empty:
            continue
        q0 = after_grp['Дебит нефти за последний месяц, т/сут'].max()

        # Параметры
        match = params_df[params_df['Скважина'] == skv]
        if match.empty:
            continue
        perm = match['Проницаемость'].values[0]
        h = match['Толщина'].values[0]
        x = skv_data['Координата X'].iloc[0]
        y = skv_data['Координата Y'].iloc[0]

        results.append({
            'Скважина': skv,
            'X': x,
            'Y': y,
            'Проницаемость': perm,
            'Толщина': h,
            'q_before': q_before,
            'q0': q0
        })

    return pd.DataFrame(results)

# Скважина, X, Y, Проницаемость, Толщина, q_before, q0
known_q0_df = get_actual_q0_after_grp(wells_df, grp_df, params_df)

from sklearn.metrics.pairwise import euclidean_distances

def predict_q0_for_non_grp_wells(wells_df, params_df, known_q0_df, k=5):
    wells_df = wells_df.copy()
    wells_df['Скважина'] = wells_df['№ скважины'].astype(str).str.strip().str.upper()
    wells_df['Дата'] = pd.to_datetime(wells_df['Дата'], dayfirst=True)
    params_df['Скважина'] = params_df['Скважина'].astype(str).str.strip().str.upper()

    used_grp = set(known_q0_df['Скважина'])
    all_well_ids = wells_df['Скважина'].unique()
    predict_ids = [wid for wid in all_well_ids if wid not in used_grp]

    pred_results = []

    for skv in predict_ids:
        skv_data = wells_df[wells_df['Скважина'] == skv].sort_values('Дата')
        if skv_data.empty:
            continue

        recent = skv_data[skv_data['Дебит нефти за последний месяц, т/сут'] > 0]
        if recent.empty:
            continue
        q_before = recent['Дебит нефти за последний месяц, т/сут'].iloc[-1]

        # Параметры
        match = params_df[params_df['Скважина'] == skv]
        if match.empty:
            continue
        perm = match['Проницаемость'].values[0]
        h = match['Толщина'].values[0]
        x = skv_data['Координата X'].iloc[0]
        y = skv_data['Координата Y'].iloc[0]

        # Считаем расстояние в 5D-пространстве
        target_vector = np.array([[x, y, perm, h, q_before]])
        base_matrix = known_q0_df[['X', 'Y', 'Проницаемость', 'Толщина', 'q_before']].values

        distances = euclidean_distances(target_vector, base_matrix)[0]
        if np.sum(distances == 0) > 0:
            distances[distances == 0] = 1e-6  # чтобы не делить на ноль

        weights = 1 / distances
        weights /= weights.sum()

        q0_pred = np.sum(weights * known_q0_df['q0'].values)

        pred_results.append({
            'Скважина': skv,
            'X': x,
            'Y': y,
            'Проницаемость': perm,
            'Толщина': h,
            'q_before': q_before,
            'q0_predicted': q0_pred
        })

    return pd.DataFrame(pred_results)

# Предсказание q0 по аналогам
predicted_q0_df = predict_q0_for_non_grp_wells(wells_df, params_df, known_q0_df, k=5)

predicted_q0_df.to_excel("Предсказанные_запускные_дебиты_после_ГРП.xlsx", index=False)

from sklearn.ensemble import RandomForestRegressor

# Признаки: координаты, проницаемость, толщина, дебит до ГРП
X = known_q0_df[['X', 'Y', 'Проницаемость', 'Толщина', 'q_before']]

# Целевая переменная — запускной дебит после ГРП
y = known_q0_df['q0']

# Обучение модели
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Входные признаки для скважин, где ГРП не было
X_new = predicted_q0_df[['X', 'Y', 'Проницаемость', 'Толщина', 'q_before']]

# Предсказание запускного дебита после гипотетического ГРП
q0_preds = model.predict(X_new)

# Добавим это к датафрейму
predicted_q0_df['q0_predicted_rf'] = q0_preds


import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.hist(predicted_q0_df['q0_predicted_rf'] * 0.9, bins=20, alpha=0.6, label='Random Forest')
plt.xlabel('Предсказанный запускной дебит жидкости, т/сут')
plt.ylabel('Число скважин')
plt.legend()
plt.grid()
plt.title('Распределение предсказанных дебитов')
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt
import seaborn as sns

# Получи датафрейм с q0
known_q0_df = get_actual_q0_after_grp(wells_df, grp_df, params_df)

# Сортируем по q0
plot_df = known_q0_df.sort_values('q0', ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x='Скважина', y='q0', data=plot_df, palette='viridis')

plt.xticks(rotation=90)
plt.xlabel("Скважина (с проведённым ГРП)")
plt.ylabel("Запускной дебит по нефти, т/сут")
plt.title("Фактические запускные дебиты нефти после ГРП")
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.xticks([])
plt.show()

# Сохраняем предсказания по Random Forest отдельно
columns_to_export_rf = ['Скважина', 'Проницаемость', 'Толщина', 'q_before', 'q0_predicted_rf']
predicted_q0_df[columns_to_export_rf].to_excel("Предсказанные_дебиты_ИИ_RF.xlsx", index=False)


## Расчет дебитов после ГТМ

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def generate_rb_forecasts(arps_params_path, rb_start_debit_path, df_dynamic, mode='нефть'):
    if mode not in ('нефть', 'жидкость'):
        raise ValueError("Параметр mode должен быть 'нефть' или 'жидкость'.")

    forecast_col = 'Прогноз дебита нефти, т/сут' if mode == 'нефть' else 'Прогноз дебита жидкости, т/сут'
    output_dir = "РБ_динамика_нефть" if mode == 'нефть' else "РБ_динамика_жидкость"

    df_params = pd.read_excel(arps_params_path)
    df_rb = pd.read_excel(rb_start_debit_path)

    df_params['Скважина'] = df_params['Скважина'].astype(str).str.strip().str.upper()
    df_rb['Скважина'] = df_rb['Скважина'].astype(str).str.strip().str.upper()

    merged = pd.merge(df_params, df_rb, on='Скважина', how='inner')
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    all_forecasts = []

    for _, row in merged.iterrows():
        skv = row['Скважина']
        q0 = row['Запускной дебит нефть'] if mode == 'нефть' else row['Запускной дебит жидкость']
        d = row['d']
        b = row['b']
        if pd.isna(q0) or pd.isna(d) or pd.isna(b):
            print(f"Пропуск скважины {skv} из-за отсутствующих данных.")
            continue

        t_vals = np.arange(0, 300)
        decline_curve = arps_func(t_vals, d, b)
        forecast = q0 * decline_curve
        q_result = []

        # дебит базы
        dynamic_data = df_dynamic[df_dynamic['Скважина'] == skv]
        if dynamic_data.empty:
            continue

        for q, t in zip(forecast, t_vals):
            if t - 1 >= len(dynamic_data):
                q_result.append(q)
                continue
            existing = dynamic_data.iloc[t - 1][forecast_col]
            if pd.isna(existing):
                q_result.append(q)
            else:
                q_result.append(max(q, existing))

        df_forecast = pd.DataFrame({
            'Скважина': skv,
            'Месяц': t_vals,
            forecast_col: q_result
        })

        all_forecasts.append(df_forecast)

        file_path = Path(output_dir) / f"{skv}_прогноз_RB.xlsx"
        df_forecast.to_excel(file_path, index=False)
        print(f"Сохранён прогноз для скважины {skv} в файл: {file_path}")

    # Объединяем все прогнозы в один DataFrame
    final_df = pd.concat(all_forecasts, ignore_index=True)
    return final_df

# Пример вызова:
q_liquid_after_gtm = generate_rb_forecasts(
    arps_params_path=r"C:\Users\pogre\Desktop\Проект для НТК\Темпы окружения по жидкости\Arps_параметры.xlsx",
    rb_start_debit_path=r"РБ_запуск.xlsx",
    df_dynamic=df_liquid_prognoz_baza,
    mode='жидкость'  # или 'нефть'
)

q_oil_after_gtm = generate_rb_forecasts(
    arps_params_path=r"C:\Users\pogre\Desktop\Проект для НТК\Темпы окружения по нефти\Arps_параметры.xlsx",
    rb_start_debit_path= r"РБ_запуск.xlsx",
    df_dynamic=df_oil_prognoz_baza, 
    mode='нефть'
)


### Расчёт приростов

In [ ]:
import pandas as pd

def calculate_monthly_gain_aligned(df_after_gtm, df_baza, value_col, output_path):
    # Убедимся, что оба датафрейма содержат нужные столбцы
    required_columns = {'Скважина', 'Месяц', value_col}
    if not required_columns.issubset(df_after_gtm.columns) or not required_columns.issubset(df_baza.columns):
        raise ValueError(f"Оба датафрейма должны содержать столбцы: {required_columns}")
    print(q_liquid_after_gtm)
    # Объединяем датафреймы по скважине и месяцу
    merged = pd.merge(
        df_after_gtm[['Скважина', 'Месяц', value_col]],
        df_baza[['Скважина', 'Месяц', value_col]],
        on=['Скважина', 'Месяц'],
        suffixes=('_after', '_baza')
    )
    print(merged)
    # Вычисляем прирост
    merged['Прирост'] = merged[f'{value_col}_after'] - merged[f'{value_col}_baza']
    merged['Прирост'] = merged['Прирост'].clip(lower=0)

    # Сохраняем результат
    result = merged[['Скважина', 'Месяц', 'Прирост']]
    result.to_excel(output_path, index=False)

    print(f"Приросты по месяцам сохранены в файл: {output_path}")
    return result

q_oil_prirost = calculate_monthly_gain_aligned(df_after_gtm=q_oil_after_gtm,
                       df_baza=df_oil_prognoz_baza,
                       value_col='Прогноз дебита нефти, т/сут',
                       output_path='Приросты_нефти_по_месяцам.xlsx')

q_liquid_prirost = calculate_monthly_gain_aligned(df_after_gtm=q_liquid_after_gtm,
                       df_baza=df_liquid_prognoz_baza,
                       value_col='Прогноз дебита жидкости, т/сут',
                       output_path='Приросты_жидкости_по_месяцам.xlsx')


### Расчёт суммарных приростов

In [ ]:
import pandas as pd

def calculate_total_gains(df_gain_oil, df_gain_liq,
                          start_month=0, end_month=None,
                          output_path='Суммарные_приросты.xlsx'):
    """
    df_gain_oil: датафрейм с колонками ['Скважина', 'Месяц', 'Прирост'] по нефти
    df_gain_liq: датафрейм с колонками ['Скважина', 'Месяц', 'Прирост'] по жидкости
    start_month: начальный месяц (включительно)
    end_month: конечный месяц (включительно); если None — берется максимум по каждому df
    """

    def process_gain(df, label):
        df = df.copy()
        if end_month is None:
            local_end = df['Месяц'].max()
        else:
            local_end = end_month
        df = df[(df['Месяц'] >= start_month) & (df['Месяц'] <= local_end)]
        result = df.groupby('Скважина')['Прирост'].sum().reset_index()
        result[f'Суммарный прирост {label}, м3'] = result['Прирост'] * 30
        return result[['Скважина', f'Суммарный прирост {label}, м3']]

    oil_result = process_gain(df_gain_oil, 'нефти')
    liq_result = process_gain(df_gain_liq, 'жидкости')

    # Объединение по скважинам
    result = pd.merge(oil_result, liq_result, on='Скважина', how='outer')

    # Сохранение
    result.to_excel(output_path, index=False)
    print(f"Суммарные приросты сохранены в файл: {output_path}")

    return result

calculate_total_gains(
    df_gain_oil=q_oil_prirost,
    df_gain_liq=q_liquid_prirost,
    start_month=0,
    end_month=96,
    output_path='суммарные_приросты_0_6.xlsx'
)
